# Closed-loop uniformity correction for the light sheet (issue #23)

Per-iteration workflow:
1. Push current payload to the SLM hardware and pull the camera bmp.
2. Analyze the bmp via `slm.analyze_sheet` and extract the flat-region profile `v`.
3. Every `STEPS` iterations, average the collected `v` vectors, compute reweight `w`, and refresh `payload/sheet/testfile_sheet_payload.npz`.

This notebook **requires GS Windows hardware** to be reachable via `push_run.sh`. Cells 1–3 (imports / config / helpers) and the initial payload-regen cell can run anywhere; the loop cells will block on the hardware roundtrip.

In [ ]:
from __future__ import annotations
import json, os, platform, shutil, subprocess, sys, time
from pathlib import Path

import numpy as np
import torch
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt

from slm import analyze_sheet, plot_sheet_analysis
from slm.cgm import CGM_phase_generate
from slm.generation import SLM_class
from slm import imgpy
from slm.metrics import fidelity as _fidelity, efficiency as _efficiency
from slm.propagation import fft_propagate
from slm.targets import measure_region as _measure_region

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent
print('REPO_ROOT =', REPO_ROOT)

In [ ]:
# ----- Hyperparameters -----
TOTAL_LOOP = 10      # push+analyze iterations
STEPS      = 1       # refresh payload every STEPS iters
STEEPNESS  = 0.2     # 0: no correction; 1: w = sqrt(1/v). Kept small + clipped.
CLIP_LO    = 0.85
CLIP_HI    = 1.15
FLAT_A_UM  = 50.0
FLAT_B_UM  = 175.0

SLM_FLAT_WIDTH_PX = 35
SLM_GAUSS_SIGMA   = 2
SLM_EDGE_SIGMA    = 5
TARGET_SHIFT_FPX  = 50
FRESNEL_SD        = 1000
LUT               = 207
ARRAY_BIT         = 12
CGM_MAX_ITER      = 1000
CGM_STEEPNESS     = 9
CGM_ETA_STEEPNESS = 8
SETTING_ETA       = 0.1
BEAM_CENTER_DX_UM = 0
BEAM_CENTER_DY_UM = 0

PUSH_RETRY_COUNT = 3
PUSH_RETRY_SLEEP = 10

IS_WINDOWS = platform.system() == 'Windows'
AFTER_BMP  = REPO_ROOT / 'data' / 'sheet' / 'testfile_sheet_after.bmp'
OUTPUT_DIR = REPO_ROOT / 'payload' / 'sheet'
PAYLOAD    = OUTPUT_DIR / 'testfile_sheet_payload.npz'
PARAMS_PATH = OUTPUT_DIR / 'testfile_sheet_params.json'
PUSH_RUN   = REPO_ROOT / ('push_run.ps1' if IS_WINDOWS else 'push_run.sh')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ----- Helpers -----
def compute_reweight(v: np.ndarray, steepness: float,
                     clip_lo: float = CLIP_LO, clip_hi: float = CLIP_HI) -> np.ndarray:
    """Amplitude-domain correction, steepness-blended and hard-clipped."""
    v = np.asarray(v, dtype=np.float64)
    v_norm = np.clip(v / v.mean(), 1e-6, None)
    inv_sqrt = 1.0 / np.sqrt(v_norm)
    inv_sqrt = inv_sqrt / inv_sqrt.mean()
    w = (1.0 - steepness) + steepness * inv_sqrt
    w = np.clip(w, clip_lo, clip_hi)
    return w / w.mean()


def push_run_retry(payload_arg: str) -> None:
    """Call push_run.{sh,ps1} with retry; transient SSH drops don't waste the CGM work."""
    for attempt in range(PUSH_RETRY_COUNT):
        try:
            if IS_WINDOWS:
                cmd = ['powershell', '-ExecutionPolicy', 'Bypass', '-File',
                       str(PUSH_RUN), payload_arg]
            else:
                cmd = ['bash', str(PUSH_RUN), payload_arg]
            subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)
            return
        except subprocess.CalledProcessError as e:
            if attempt == PUSH_RETRY_COUNT - 1:
                raise
            print(f'[RETRY] push_run attempt {attempt+1}/{PUSH_RETRY_COUNT} '
                  f'failed (exit {e.returncode}); backing off {PUSH_RETRY_SLEEP}s')
            time.sleep(PUSH_RETRY_SLEEP)

In [ ]:
# ----- Payload regeneration (was testfile_sheet.main) -----
# Inlines the CGM compute path so this notebook is self-contained.
def regenerate_payload(reweight: np.ndarray | None = None) -> dict:
    grid_px = 1 << ARRAY_BIT
    SLM = SLM_class()
    SLM.arraySizeBit = [ARRAY_BIT, ARRAY_BIT]
    SLM.image_init(
        initGaussianPhase_user_defined=np.zeros((grid_px, grid_px)), Plot=False,
        beam_center_um=(BEAM_CENTER_DX_UM, BEAM_CENTER_DY_UM),
    )
    W, H = SLM.SLMRes
    cx, cy = W // 2, H // 2
    correct = lambda screen: imgpy.SLM_screen_Correct(
        screen, LUT=LUT, correctionImgPath='calibration/CAL_LSH0905549_1013nm.bmp')
    target_center = ((SLM.ImgResY - 1) / 2.0 - TARGET_SHIFT_FPX,
                     (SLM.ImgResX - 1) / 2.0 - TARGET_SHIFT_FPX)
    targetAmp = SLM.light_sheet_target(
        flat_width=SLM_FLAT_WIDTH_PX, gaussian_sigma=SLM_GAUSS_SIGMA,
        angle=0, edge_sigma=SLM_EDGE_SIGMA, center=target_center,
        reweight=reweight,
    )
    init_phi = SLM.stationary_phase_sheet(
        flat_width=SLM_FLAT_WIDTH_PX, gaussian_sigma=None, angle=0, center=target_center,
    )
    t0 = time.perf_counter()
    cgm_device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'[CGM] {CGM_MAX_ITER} iters on {SLM.ImgResY}x{SLM.ImgResX} (device={cgm_device})')
    SLM_Phase = CGM_phase_generate(
        torch.tensor(SLM.initGaussianAmp), torch.from_numpy(init_phi),
        torch.from_numpy(targetAmp),
        max_iterations=CGM_MAX_ITER, steepness=CGM_STEEPNESS,
        eta_min=SETTING_ETA, eta_steepness=CGM_ETA_STEEPNESS, Plot=False,
    )
    cgm_wall_time = time.perf_counter() - t0
    phase_np = SLM_Phase.cpu().clone().numpy()
    phase_wrapped = np.angle(np.exp(1j * phase_np))
    SLM_screen_raw = SLM.phase_to_screen(phase_wrapped)
    fresnel = SLM.fresnel_lens_phase_generate(FRESNEL_SD, cx, cy)[0]
    SLM_screen_shift = ((SLM_screen_raw.astype(np.int32) + fresnel.astype(np.int32)) % 256).astype(np.uint8)
    SLM_screen_final = correct(SLM_screen_shift)
    region_np = _measure_region(targetAmp.shape, targetAmp, margin=5)
    E_out = fft_propagate(SLM.initGaussianAmp * np.exp(1j * phase_wrapped))
    F = float(_fidelity(E_out, targetAmp, region_np))
    eta = float(_efficiency(E_out, region_np))
    np.savez_compressed(PAYLOAD, slm_screen=SLM_screen_final)
    print(f'[SAVE] {PAYLOAD} ({PAYLOAD.stat().st_size//1024} KB)  F={F:.4f} eta={100*eta:.2f}%')
    return {
        'fidelity': F, 'efficiency': eta, 'cgm_wall_time_s': cgm_wall_time,
        'slm_screen': SLM_screen_final, 'targetAmp': targetAmp, 'E_out': E_out,
        'region': region_np,
    }

In [ ]:
# Initial payload generation. Run this once before entering the loop.
ts = time.strftime('%Y%m%d_%H%M%S', time.localtime())
out_dir = REPO_ROOT / 'data' / f'sheet_inloop_{ts}_a{int(FLAT_A_UM)}_b{int(FLAT_B_UM)}'
out_dir.mkdir(parents=True, exist_ok=True)
print(f'[INLOOP] total_loop={TOTAL_LOOP} steps={STEPS} steepness={STEEPNESS} '
      f'clip=[{CLIP_LO},{CLIP_HI}] flat=[{FLAT_A_UM},{FLAT_B_UM}]um')
print(f'[INLOOP] output dir: {out_dir}')
_ = regenerate_payload(reweight=None)

In [ ]:
# ----- Single iteration (manual step) -----
# Re-evaluate this cell to advance the loop one step at a time. Useful for
# debugging the per-iter behavior. Replace with the batch cell below for
# unattended runs.
flat_buffer: list[np.ndarray] = []
history: list[dict] = []
last_w: np.ndarray | None = None
i = len(history)

payload_arg = str(PAYLOAD.relative_to(REPO_ROOT))
push_run_retry(payload_arg)
iter_bmp = out_dir / f'iter_{i:02d}.bmp'
shutil.copy2(AFTER_BMP, iter_bmp)

result = analyze_sheet(iter_bmp, flat_a=FLAT_A_UM, flat_b=FLAT_B_UM)
fig = plot_sheet_analysis(result)
plt.show()

flat_buffer.append(np.asarray(result['flat_profile'], dtype=np.float64))
history.append({
    'iter': i,
    'rms_percent': result['rms_percent'],
    'pk_pk_percent': result['pk_pk_percent'],
    'efficiency_observed': result['efficiency_observed'],
    'flat_mean': result['flat_top_mean_intensity'],
    'reweight_was_applied': last_w is not None,
})
print(f"[ITER {i:02d}] rms={result['rms_percent']:.3f}%  "
      f"pkpk={result['pk_pk_percent']:.3f}%  "
      f"eff={100*result['efficiency_observed']:.3f}%")

In [ ]:
# ----- Batch loop (all TOTAL_LOOP iterations) -----
flat_buffer = []
history = []
last_w = None

for i in range(TOTAL_LOOP):
    payload_arg = str(PAYLOAD.relative_to(REPO_ROOT))
    push_run_retry(payload_arg)
    iter_bmp = out_dir / f'iter_{i:02d}.bmp'
    shutil.copy2(AFTER_BMP, iter_bmp)

    result = analyze_sheet(iter_bmp, flat_a=FLAT_A_UM, flat_b=FLAT_B_UM)
    flat = np.asarray(result['flat_profile'], dtype=np.float64)
    flat_buffer.append(flat)
    history.append({
        'iter': i,
        'rms_percent': result['rms_percent'],
        'pk_pk_percent': result['pk_pk_percent'],
        'efficiency_observed': result['efficiency_observed'],
        'flat_mean': result['flat_top_mean_intensity'],
        'reweight_was_applied': last_w is not None,
    })
    print(f"[ITER {i:02d}] rms={result['rms_percent']:.3f}%  "
          f"pkpk={result['pk_pk_percent']:.3f}%  "
          f"eff={100*result['efficiency_observed']:.3f}%")

    if (i + 1) % STEPS == 0 and (i + 1) < TOTAL_LOOP:
        v_avg = np.mean(np.stack(flat_buffer), axis=0)
        w = compute_reweight(v_avg, STEEPNESS)
        print(f'[UPDATE] v_avg range=[{(v_avg/v_avg.mean()).min():.3f}, '
              f'{(v_avg/v_avg.mean()).max():.3f}]  -> '
              f'w range=[{w.min():.3f}, {w.max():.3f}]  (clip={CLIP_LO}/{CLIP_HI})')
        last_w = w
        regenerate_payload(reweight=w)
        flat_buffer.clear()

summary = {
    'timestamp': ts,
    'hyperparams': {
        'total_loop': TOTAL_LOOP, 'steps': STEPS, 'steepness': STEEPNESS,
        'flat_a_um': FLAT_A_UM, 'flat_b_um': FLAT_B_UM,
        'slm_flat_width_px': SLM_FLAT_WIDTH_PX, 'slm_gauss_sigma': SLM_GAUSS_SIGMA,
    },
    'iterations': history,
    'final_reweight': None if last_w is None else last_w.tolist(),
}
with open(out_dir / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f"[INLOOP] done. summary: {out_dir / 'summary.json'}")

In [ ]:
# ----- RMS / Pk-Pk progression plot -----
if history:
    iters = [h['iter'] for h in history]
    rms = [h['rms_percent'] for h in history]
    pkpk = [h['pk_pk_percent'] for h in history]
    eff = [100*h['efficiency_observed'] for h in history]
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(iters, rms, 'o-', label='RMS %')
    ax[0].plot(iters, pkpk, 's-', label='Pk-Pk %')
    ax[0].set_xlabel('iter'); ax[0].set_ylabel('%'); ax[0].legend(); ax[0].grid(alpha=0.3)
    ax[0].set_title('Uniformity vs iteration')
    ax[1].plot(iters, eff, 'd-', color='tab:green')
    ax[1].set_xlabel('iter'); ax[1].set_ylabel('observed efficiency %')
    ax[1].grid(alpha=0.3); ax[1].set_title('Observed efficiency vs iteration')
    plt.tight_layout(); plt.show()